# Streaming training with LSDB datasets

## Open a TESS lightcurve catalog
We'll open the TESS lightcurve catalog from LSDB and filter down the observations
to non-NaN values and ensure that the lightcurves have at least 100 observations.


In [ ]:
import lsdb

tess = lsdb.open_catalog("https://data.lsdb.io/hats/tess/tess_lightcurve")

In [ ]:
def drop_nans(df):
    # 1. Drop the rows where lightcurve.sap_flux is NaN
    # 2. Also remove all the resulting objects with no lightcurve points remaining
    df["lightcurve.sap_flux"] = df["lightcurve.sap_flux"].astype(float)
    return df.dropna(subset=["lightcurve.sap_flux"]).dropna(subset=["lightcurve"])


tess_filtered_nans = tess.map_partitions(drop_nans)

from nested_pandas.utils import count_nested


def count_points(pts):
    # Asked to count `lightcurve`, this will add a column called `n_lightcurve`
    return count_nested(pts, "lightcurve")


min_observations = 100
tess_filtered_nans = tess_filtered_nans.map_partitions(count_points).query(
    f"n_lightcurve >= {min_observations}"
)

## Hyrax section
Instantiate Hyrax and define the data request using TessLSDBStreamDataset.
This dataset implements two collation functions for the time and sap_flux columns.

In [ ]:
from hyrax import Hyrax
from hyrax.datasets import LSDBStreamDataset

h = Hyrax()

In [ ]:
LSDBStreamDataset.register_catalog("tess_catalog", tess_filtered_nans)
data_request = {
    "train_stream": {
        "data": {
            "dataset_class": "TessLSDBStreamDataset",
            "data_location": "lsdb://tess_catalog",
            "primary_id_field": "ticid",
            "fields": ["ra_obj", "dec_obj", "lightcurve_time", "lightcurve_sap_flux"],
        }
    }
}
h.set_config("data_request", data_request)
h.set_config("model.name", "HyraxLoopback")

In [ ]:
batch_counter = 0
MAX_BATCHES = 10

with h.train_stream() as session:
    for batch, metrics in session:
        print(metrics)
        batch_counter += 1

        if batch_counter >= MAX_BATCHES:
            break